In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog_name", "ecommerce", "Catalog Name")
dbutils.widgets.text("storage_account_name", "ecommcistorage05", "Storage Account Name")
dbutils.widgets.text("container_name", "ecomm-raw-data", "Container Name")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
storage_account_name = dbutils.widgets.get("storage_account_name")
container_name = dbutils.widgets.get("container_name")

In [0]:
df_batch = spark.read.table(f"{catalog_name}.bronze.brz_returns")
display(df_batch.limit(20))

return_ts,order_dt,order_id,reason,_rescued_data,ingest_timestamp,source_file
2024-05-01T00:02:45.000Z,2024-04-20,122138,Wrong Item,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T01:27:25.000Z,2024-04-21,123994,Wrong Item,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T02:24:11.000Z,2024-04-19,120440,Other,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T02:45:13.000Z,2024-04-21,123548,Quality Issue,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T03:12:47.000Z,2024-04-21,123925,Wrong Item,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T03:32:46.000Z,2024-04-19,120365,Other,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T04:24:57.000Z,2024-04-16,117331,Late Delivery,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T05:33:52.000Z,2024-04-21,123702,Late Delivery,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T07:06:09.000Z,2024-04-18,119335,Late Delivery,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv
2024-05-01T07:15:25.000Z,2024-04-20,122387,Damaged,null,2026-09-27T13:38:00.689Z,abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/order_returns/landing/returns_2024-05.csv


In [0]:
df = spark.readStream.format("delta").table(f"{catalog_name}.bronze.brz_returns")

### Remove duplicates based on `order_id`, `order_dt`, `return_ts`

In [0]:
df = df.dropDuplicates(["order_id", "return_ts","order_dt"])

### Convert `order_dt` to DateType.

In [0]:
df = df.withColumn("order_dt", F.to_date(F.col("order_dt")))

### Convert `return_ts` to TimestampType.

In [0]:
df = df.withColumn("return_ts", F.to_timestamp(F.col("return_ts")))

### Convert `reason` column to uppercase and trim whitespace

In [0]:
df = df.withColumn("reason",F.upper(F.trim(F.col("reason"))))

### Add `processed_time` column with the current timestamp.

In [0]:
df = df.withColumn("processed_time",F.current_timestamp())

### Saving the table in silver layer

In [0]:
# Silver checkpoint
silver_checkpoint_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/checkpoint/silver/fact_returns/"
print(silver_checkpoint_path)

abfss://ecomm-raw-data@stgecomadlsddevap001.dfs.core.windows.net/checkpoint/silver/fact_returns/


In [0]:
df_batch.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_batch.columns
]).show()


+---------+--------+--------+------+-------------+----------------+-----------+
|return_ts|order_dt|order_id|reason|_rescued_data|ingest_timestamp|source_file|
+---------+--------+--------+------+-------------+----------------+-----------+
|        0|       0|       0|     0|        20194|               0|          0|
+---------+--------+--------+------+-------------+----------------+-----------+



In [0]:
# checking which column can be considered as primary key.
def check_key(df, cols):
    total = df.count()
    distinct = df.select(*cols).distinct().count()
    nulls = df.filter(" OR ".join([f"{c} IS NULL" for c in cols])).count()
    print(f"{cols} -> total={total}, distinct={distinct}, nulls={nulls}, "
          f"is_unique={total == distinct}, no_nulls={nulls == 0}")

In [0]:
check_key(df_batch, ["order_id"])
check_key(df_batch, ["order_id", "return_ts"])

# see the offenders if orderid isn't unique
df_batch.groupBy("order_id").count().filter("count > 1").show()

['order_id'] -> total=20194, distinct=20194, nulls=0, is_unique=True, no_nulls=True
['order_id', 'return_ts'] -> total=20194, distinct=20194, nulls=0, is_unique=True, no_nulls=True
+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [0]:
def upsert_to_silver(microBatchDF, batchId):
    table_name = f"{catalog_name}.silver.slv_returns"
    if not spark.catalog.tableExists(table_name):
        print("creating new table")
        microBatchDF.write.format("delta").mode("overwrite").saveAsTable(table_name)
        spark.sql(
            f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
        )
    else:
        deltaTable = DeltaTable.forName(spark, table_name)
        deltaTable.alias("silver_table").merge(
            microBatchDF.alias("batch_table"),
            "silver_table.order_id = batch_table.order_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()    

In [0]:
df.writeStream.trigger(availableNow=True).foreachBatch(
    upsert_to_silver
).format("delta").option("checkpointLocation", silver_checkpoint_path).option(
    "mergeSchema", "true"
).outputMode(
    "update"
).trigger(
    once=True
).start().awaitTermination()